# Alfvén (1942) — Existence of Electromagnetic-Hydrodynamic Waves: Implementation
# 전자기유체역학 파동의 존재: 구현

**Paper / 논문**: H. Alfvén, "Existence of Electromagnetic-Hydrodynamic Waves," *Nature*, 150, 405–406, 1942.

이 노트북은 Alfvén 파의 물리학을 처음부터 구현하고 시각화합니다:

This notebook implements and visualizes the physics of Alfvén waves from scratch:

1. **Part 1**: Alfvén 파 속도 계산 — 논문의 수치 재현 / Alfvén speed calculation — reproducing paper's numbers
2. **Part 2**: 파동 방정식 유도 및 해석해 시각화 / Wave equation derivation and analytical solution visualization
3. **Part 3**: 1D Alfvén 파 수치 시뮬레이션 (from scratch) / 1D numerical simulation (from scratch)
4. **Part 4**: 기타 줄 비유 — 자기장력과 파동의 관계 / Guitar string analogy — magnetic tension and waves
5. **Part 5**: 에너지 등분배 검증 / Energy equipartition verification
6. **Part 6**: 태양 환경별 Alfvén 속도 비교 / Alfvén speed across solar environments
7. **Part 7**: Frozen-in 조건 시각화 / Frozen-in condition visualization
8. **Part 8**: 현대적 맥락 — MHD 파동의 세 가지 모드 / Modern context — three MHD wave modes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Circle
from matplotlib.gridspec import GridSpec
from scipy.integrate import solve_ivp

plt.rcParams.update({
    'figure.figsize': (14, 7),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# Physical constants
MU_0 = 4 * np.pi * 1e-7  # Permeability of free space (H/m)
C_CGS = 3e10  # Speed of light (cm/s) for CGS conversion

## Part 1: Alfvén Speed Calculation — Reproducing the Paper's Numbers
## 파트 1: Alfvén 속도 계산 — 논문의 수치 재현

Alfvén의 원래 공식 (CGS): $V = \frac{H_0}{\sqrt{4\pi\partial}}$

현대 공식 (SI): $v_A = \frac{B_0}{\sqrt{\mu_0 \rho_0}}$

논문에서 Alfvén은 $H_0 = 15$ gauss, $\partial = 0.005$ g/cm³으로 $V \sim 60$ cm/s를 얻었습니다.
이 계산을 CGS와 SI 양쪽에서 재현합니다.

Alfvén obtained $V \sim 60$ cm/s with $H_0 = 15$ gauss, $\partial = 0.005$ g/cm³.
We reproduce this in both CGS and SI.

In [ ]:
def alfven_speed_cgs(H0_gauss: float, rho_cgs: float) -> float:
    """Compute Alfvén speed using Alfvén's original CGS formula.

    Args:
        H0_gauss: Magnetic field strength in gauss.
        rho_cgs: Mass density in g/cm³.

    Returns:
        Alfvén speed in cm/s.
    """
    return H0_gauss / np.sqrt(4 * np.pi * rho_cgs)


def alfven_speed_si(B0_tesla: float, rho_si: float) -> float:
    """Compute Alfvén speed using modern SI formula.

    Args:
        B0_tesla: Magnetic field strength in Tesla.
        rho_si: Mass density in kg/m³.

    Returns:
        Alfvén speed in m/s.
    """
    return B0_tesla / np.sqrt(MU_0 * rho_si)


# === Alfvén's original values ===
H0 = 15  # gauss
rho = 0.005  # g/cm³

v_cgs = alfven_speed_cgs(H0, rho)
print("=" * 60)
print("Reproducing Alfvén's calculation (CGS)")
print("=" * 60)
print(f"H₀ = {H0} gauss")
print(f"ϱ  = {rho} g/cm³")
print(f"V  = H₀/√(4πϱ) = {H0}/√(4π × {rho})")
print(f"   = {H0}/{np.sqrt(4*np.pi*rho):.4f}")
print(f"   = {v_cgs:.1f} cm/s  ✓ (Alfvén got ~60 cm/s)")

# === SI conversion ===
B0_si = H0 * 1e-4  # 1 gauss = 1e-4 T
rho_si = rho * 1e3  # 1 g/cm³ = 1e3 kg/m³

v_si = alfven_speed_si(B0_si, rho_si)
print(f"\n{'=' * 60}")
print("Same calculation in SI units")
print("=" * 60)
print(f"B₀ = {B0_si:.4f} T  (= {H0} gauss × 10⁻⁴)")
print(f"ρ  = {rho_si} kg/m³  (= {rho} g/cm³ × 10³)")
print(f"vₐ = B₀/√(μ₀ρ) = {v_si:.4f} m/s = {v_si*100:.1f} cm/s  ✓")

print(f"\n{'=' * 60}")
print("Physical context / 물리적 맥락")
print("=" * 60)
print(f"Alfvén speed: {v_cgs:.1f} cm/s = {v_cgs/1e5:.6f} km/s")
print(f"This matches sunspot zone drift speed (~0.6 m/s)")
print(f"Location: ~10⁵ km below solar surface (deep convection zone)")

## Part 2: Analytical Solution — Traveling Alfvén Wave
## 파트 2: 해석해 — 진행하는 Alfvén 파

파동 방정식 $\frac{\partial^2 \delta B}{\partial t^2} = v_A^2 \frac{\partial^2 \delta B}{\partial z^2}$의 일반해는
$\delta B(z,t) = f(z - v_A t) + g(z + v_A t)$입니다.

정현파 해: $\delta B_x(z,t) = B_1 \sin(kz - \omega t)$, $\delta v_x(z,t) = -\frac{B_1}{\sqrt{\mu_0 \rho}} \sin(kz - \omega t)$

여기서 $\omega = k \cdot v_A$ (비분산 관계).

The general solution of the wave equation is $\delta B(z,t) = f(z - v_A t) + g(z + v_A t)$.
Sinusoidal: $\delta B_x = B_1 \sin(kz - \omega t)$, $\delta v_x = -\frac{B_1}{\sqrt{\mu_0\rho}} \sin(kz - \omega t)$
with $\omega = k v_A$ (non-dispersive).

In [ ]:
# Alfvén wave analytical solution visualization
# Normalized units: B0 = 1, rho = 1, mu0 = 1 → vA = 1

B0 = 1.0  # Background field (normalized)
rho0 = 1.0
va = B0 / np.sqrt(1.0 * rho0)  # vA = 1 in normalized units
B1 = 0.1  # Perturbation amplitude
k = 2 * np.pi  # Wavenumber (one wavelength in domain [0, 1])
omega = k * va  # Angular frequency

z = np.linspace(0, 2, 500)
times = [0, 0.125, 0.25, 0.375, 0.5]

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

colors = plt.cm.viridis(np.linspace(0, 0.8, len(times)))

for t_val, color in zip(times, colors):
    # Magnetic perturbation: δBx(z, t)
    dB = B1 * np.sin(k * z - omega * t_val)
    # Velocity perturbation: δvx(z, t) = -(B1/√(μ₀ρ)) sin(kz - ωt)
    dv = -(B1 / np.sqrt(1.0 * rho0)) * np.sin(k * z - omega * t_val)

    axes[0].plot(z, dB, color=color, lw=2, label=f't = {t_val:.3f}')
    axes[1].plot(z, dv, color=color, lw=2, label=f't = {t_val:.3f}')

# Total field lines (Bz + δBx gives tilted field lines)
ax2 = axes[2]
t_snapshot = 0.0
n_lines = 20
z_fine = np.linspace(0, 2, 200)
for i in range(n_lines):
    x_base = -0.15 + i * 0.3 / n_lines
    dB_line = B1 * np.sin(k * z_fine - omega * t_snapshot)
    x_displaced = x_base + dB_line * 0.5  # Scale for visibility
    ax2.plot(z_fine, x_displaced, 'b-', lw=0.8, alpha=0.5)
# Highlight one field line
x_base_hl = 0.0
dB_hl = B1 * np.sin(k * z_fine - omega * t_snapshot)
ax2.plot(z_fine, x_base_hl + dB_hl * 0.5, 'r-', lw=2.5, label='Perturbed field line')
ax2.axhline(0, color='gray', ls='--', lw=0.5)

axes[0].set_ylabel('δBₓ / B₁', fontsize=13)
axes[0].set_title('Magnetic Perturbation / 자기장 섭동 δBₓ(z, t)', fontsize=13)
axes[0].legend(fontsize=9, ncol=5)

axes[1].set_ylabel('δvₓ / (B₁/√μ₀ρ)', fontsize=13)
axes[1].set_title('Velocity Perturbation / 속도 섭동 δvₓ(z, t)', fontsize=13)
axes[1].legend(fontsize=9, ncol=5)

ax2.set_ylabel('x (transverse)', fontsize=13)
ax2.set_xlabel('z (along B₀) / wavelength', fontsize=13)
ax2.set_title('Perturbed Field Lines at t=0 / t=0에서의 자기장 선', fontsize=13)
ax2.legend(fontsize=10)
ax2.set_ylim(-0.15, 0.15)

plt.suptitle("Alfvén Wave: Analytical Solution (Normalized Units)\n"
             "Alfvén 파: 해석해 (정규화 단위)", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("Key features / 핵심 특성:")
print("• δBx and δvx are in anti-phase (negative correlation for +z propagation)")
print("• Wave propagates along z (field direction) without changing shape")
print("• Field lines are displaced transversely — like plucked guitar strings")

## Part 3: 1D Alfvén Wave Numerical Simulation (From Scratch)
## 파트 3: 1D Alfvén 파 수치 시뮬레이션 (직접 구현)

Alfvén의 연립 방정식을 직접 시간 적분합니다:

$$\frac{\partial \delta B_x}{\partial t} = B_0 \frac{\partial \delta v_x}{\partial z}, \qquad \rho_0 \frac{\partial \delta v_x}{\partial t} = \frac{B_0}{\mu_0} \frac{\partial \delta B_x}{\partial z}$$

유한 차분(finite difference)으로 $z$ 미분을 근사하고, 시간 적분은 Leapfrog 방법을 사용합니다.
공간 미분에 중심 차분, 시간 적분에 staggered leapfrog를 사용하여 파동 방정식에 최적화된 수치 기법을 구현합니다.

We time-integrate Alfvén's coupled equations using leapfrog (staggered in time) with centered finite differences in space — an optimal scheme for wave equations.

In [ ]:
def simulate_alfven_wave(Nz: int = 400, Nt: int = 800, L: float = 2.0,
                         T_total: float = 2.0, B0: float = 1.0,
                         rho0: float = 1.0, mu0: float = 1.0,
                         amplitude: float = 0.1) -> dict:
    """Simulate 1D Alfvén wave propagation using leapfrog method.

    Solves the coupled linearized MHD equations:
        ∂(δB)/∂t = B0 * ∂(δv)/∂z
        ρ0 * ∂(δv)/∂t = (B0/μ0) * ∂(δB)/∂z

    Args:
        Nz: Number of spatial grid points.
        Nt: Number of time steps.
        L: Domain length.
        T_total: Total simulation time.
        B0: Background magnetic field.
        rho0: Background density.
        mu0: Permeability.
        amplitude: Initial perturbation amplitude.

    Returns:
        Dictionary with z, t arrays and snapshots of dB and dv.
    """
    va = B0 / np.sqrt(mu0 * rho0)
    dz = L / Nz
    dt = T_total / Nt

    # CFL condition check
    cfl = va * dt / dz
    assert cfl < 1.0, f"CFL = {cfl:.3f} > 1: unstable! Reduce dt or increase Nz."

    z = np.linspace(0, L, Nz, endpoint=False)

    # Initial condition: Gaussian pulse in δBx
    z_center = 0.5
    sigma = 0.05
    dB = amplitude * np.exp(-((z - z_center) ** 2) / (2 * sigma ** 2))
    # For a purely +z traveling wave: δv = -δB / √(μ0 ρ0)
    dv = -dB / np.sqrt(mu0 * rho0)

    # Store snapshots
    snapshot_times = [0, 0.25, 0.5, 0.75, 1.0, 1.5]
    snapshots = []
    t = 0.0

    # Half-step for leapfrog initialization (kick dv by dt/2)
    dBdz = np.zeros(Nz)
    dBdz[1:-1] = (dB[2:] - dB[:-2]) / (2 * dz)
    dBdz[0] = (dB[1] - dB[-1]) / (2 * dz)  # periodic BC
    dBdz[-1] = (dB[0] - dB[-2]) / (2 * dz)
    dv += 0.5 * dt * (B0 / (mu0 * rho0)) * dBdz

    for n in range(Nt):
        t = n * dt

        # Save snapshots
        for st in snapshot_times:
            if abs(t - st) < dt / 2 and not any(
                abs(s['t'] - st) < dt for s in snapshots
            ):
                snapshots.append({
                    't': t, 'dB': dB.copy(), 'dv': dv.copy()
                })

        # Leapfrog: update dB (full step)
        dvdz = np.zeros(Nz)
        dvdz[1:-1] = (dv[2:] - dv[:-2]) / (2 * dz)
        dvdz[0] = (dv[1] - dv[-1]) / (2 * dz)
        dvdz[-1] = (dv[0] - dv[-2]) / (2 * dz)
        dB += dt * B0 * dvdz

        # Leapfrog: update dv (full step)
        dBdz = np.zeros(Nz)
        dBdz[1:-1] = (dB[2:] - dB[:-2]) / (2 * dz)
        dBdz[0] = (dB[1] - dB[-1]) / (2 * dz)
        dBdz[-1] = (dB[0] - dB[-2]) / (2 * dz)
        dv += dt * (B0 / (mu0 * rho0)) * dBdz

    return {'z': z, 'snapshots': snapshots, 'va': va, 'L': L}


# Run simulation
result = simulate_alfven_wave()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

colors = plt.cm.plasma(np.linspace(0, 0.85, len(result['snapshots'])))

for snap, color in zip(result['snapshots'], colors):
    ax1.plot(result['z'], snap['dB'], color=color, lw=2,
             label=f"t = {snap['t']:.2f}")
    ax2.plot(result['z'], snap['dv'], color=color, lw=2,
             label=f"t = {snap['t']:.2f}")

ax1.set_ylabel('δBₓ', fontsize=13)
ax1.set_title('Numerical Simulation: Magnetic Perturbation\n'
              '수치 시뮬레이션: 자기장 섭동 (Leapfrog method)', fontsize=13)
ax1.legend(fontsize=9, ncol=3)

ax2.set_ylabel('δvₓ', fontsize=13)
ax2.set_xlabel('z (along B₀)', fontsize=13)
ax2.set_title('Velocity Perturbation / 속도 섭동', fontsize=13)
ax2.legend(fontsize=9, ncol=3)

# Mark initial position and expected position
for ax in [ax1, ax2]:
    ax.axvline(0.5, color='gray', ls=':', lw=1, alpha=0.5)
    for snap in result['snapshots']:
        expected_z = (0.5 + result['va'] * snap['t']) % result['L']
        ax.axvline(expected_z, color='red', ls=':', lw=0.5, alpha=0.3)

plt.suptitle("1D Alfvén Wave: Gaussian Pulse Propagation (From Scratch)\n"
             "1D Alfvén 파: 가우시안 펄스 전파 (직접 구현)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Alfvén speed vA = {result['va']:.1f} (normalized)")
print(f"Pulse travels from z=0.5 rightward at speed vA")
print(f"Periodic boundary: pulse wraps around at z={result['L']:.1f}")
print(f"Non-dispersive: pulse shape is preserved! (no broadening)")

## Part 4: Guitar String Analogy — Magnetic Tension as Restoring Force
## 파트 4: 기타 줄 비유 — 자기장력이 복원력

Alfvén 파의 물리적 직관: 자기장 선 = 장력을 가진 탄성 줄.

기타 줄: $v = \sqrt{T/\mu}$ (장력 $T$, 선밀도 $\mu$)
Alfvén 파: $v_A = \sqrt{B^2/(\mu_0\rho)} = B/\sqrt{\mu_0\rho}$ (자기장력 $B^2/\mu_0$, 체적밀도 $\rho$)

이 비유를 시각적으로 보여줍니다: 줄을 튕기면(자기장 선을 변위시키면) 장력이 복원력으로 작용하여 파동이 전파됩니다.

Physical intuition: magnetic field lines = elastic strings with tension.
Guitar string: $v = \sqrt{T/\mu}$; Alfvén wave: $v_A = B/\sqrt{\mu_0\rho}$.
Pluck the string (displace field line) → tension restores → wave propagates.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# --- Top Left: Guitar string at rest and plucked ---
ax = axes[0, 0]
z_str = np.linspace(0, 1, 200)
ax.plot(z_str, np.zeros_like(z_str), 'k-', lw=2, label='At rest / 정지')
# Plucked shape (triangular)
pluck = np.where(z_str < 0.4, z_str / 0.4 * 0.15,
                 np.where(z_str < 0.6, 0.15 - (z_str - 0.4) / 0.2 * 0.15,
                          0.0))
ax.plot(z_str, pluck, 'r-', lw=2.5, label='Plucked / 튕김')
# Tension arrows
for zp in [0.3, 0.5]:
    idx = np.argmin(np.abs(z_str - zp))
    slope = np.gradient(pluck, z_str)[idx]
    ax.annotate('', xy=(zp, pluck[idx] - 0.03),
                xytext=(zp, pluck[idx] + 0.01),
                arrowprops=dict(arrowstyle='->', color='blue', lw=2))
ax.text(0.35, -0.05, 'Tension T\n(restoring)', ha='center', fontsize=10, color='blue')
ax.set_title('Guitar String / 기타 줄\n$v = \\sqrt{T/\\mu}$', fontsize=13)
ax.set_ylabel('Displacement', fontsize=11)
ax.set_ylim(-0.1, 0.2)
ax.legend(fontsize=10)

# --- Top Right: Magnetic field line at rest and displaced ---
ax = axes[0, 1]
ax.plot(z_str, np.zeros_like(z_str), 'b-', lw=2, label='Equilibrium B₀')
displaced = 0.15 * np.sin(2 * np.pi * z_str)
ax.plot(z_str, displaced, 'r-', lw=2.5, label='Perturbed / 섭동')
# Magnetic tension arrows
for zp in [0.25, 0.75]:
    idx = np.argmin(np.abs(z_str - zp))
    direction = -1 if displaced[idx] > 0 else 1
    ax.annotate('', xy=(zp, displaced[idx] + direction * 0.04),
                xytext=(zp, displaced[idx]),
                arrowprops=dict(arrowstyle='->', color='purple', lw=2))
ax.text(0.5, -0.2, 'Magnetic tension $B²/\\mu_0 R$\n(restoring / 복원력)',
        ha='center', fontsize=10, color='purple')
ax.set_title('Magnetic Field Line / 자기장 선\n$v_A = B/\\sqrt{\\mu_0\\rho}$', fontsize=13)
ax.set_ylim(-0.25, 0.25)
ax.legend(fontsize=10)

# --- Bottom Left: Wave speed vs tension/field strength ---
ax = axes[1, 0]
T_range = np.linspace(0.1, 10, 100)  # Tension (normalized)
mu_string = 1.0  # Linear density
v_string = np.sqrt(T_range / mu_string)

B_range = np.linspace(0.1, 10, 100)  # Field strength (normalized)
rho_fluid = 1.0
v_alfven = B_range / np.sqrt(1.0 * rho_fluid)

ax.plot(T_range, v_string, 'k-', lw=2.5, label='String: $v = \\sqrt{T/\\mu}$')
ax.plot(B_range, v_alfven, 'b-', lw=2.5, label='Alfvén: $v_A = B/\\sqrt{\\mu_0\\rho}$')
ax.set_xlabel('Tension T  or  Field B (normalized)', fontsize=12)
ax.set_ylabel('Wave speed', fontsize=12)
ax.set_title('Wave Speed vs. Restoring Force\n파속 vs. 복원력', fontsize=13)
ax.legend(fontsize=11)

# --- Bottom Right: Comparison table ---
ax = axes[1, 1]
ax.axis('off')
table_data = [
    ['Property', 'Guitar String', 'Alfvén Wave'],
    ['Medium', 'String / 줄', 'Conducting fluid\n전도성 유체'],
    ['Restoring\nforce', 'Tension T\n장력', 'Magnetic tension\n$B^2/\\mu_0$'],
    ['Inertia', 'Linear density μ\n선밀도', 'Volume density ρ\n체적밀도'],
    ['Speed', '$v = \\sqrt{T/\\mu}$', '$v_A = B/\\sqrt{\\mu_0\\rho}$'],
    ['Wave type', 'Transverse / 횡파', 'Transverse / 횡파'],
    ['Compress?', 'No / 비압축', 'No / 비압축'],
]
table = ax.table(cellText=table_data, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.0, 2.0)
for i in range(len(table_data[0])):
    table[0, i].set_facecolor('#d4e6f1')
    table[0, i].set_text_props(fontweight='bold')
ax.set_title('Analogy Comparison / 비유 비교', fontsize=13, pad=20)

plt.suptitle("Guitar String ↔ Alfvén Wave Analogy\n"
             "기타 줄 ↔ Alfvén 파 비유", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## Part 5: Energy Equipartition Verification
## 파트 5: 에너지 등분배 검증

Alfvén 파에서 운동 에너지 밀도와 자기 에너지 밀도는 정확히 동일합니다:

$$E_k = \frac{1}{2}\rho_0\,(\delta v)^2 = \frac{(\delta B)^2}{2\mu_0} = E_B$$

이것은 Alfvén 파의 본질적 특성입니다. 수치 시뮬레이션 결과에서 이를 검증합니다.

In Alfvén waves, kinetic energy density exactly equals magnetic energy density.
We verify this from our numerical simulation.

In [ ]:
# Energy equipartition verification using analytical solution
# Normalized units: B0 = 1, rho0 = 1, mu0 = 1

B0, rho0, mu0_n = 1.0, 1.0, 1.0
va = B0 / np.sqrt(mu0_n * rho0)
B1 = 0.1
k = 2 * np.pi
omega = k * va

z = np.linspace(0, 1, 1000)
t_values = np.linspace(0, 1, 100)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# --- Top: Energy densities at t=0 ---
t = 0
dB = B1 * np.sin(k * z - omega * t)
dv = -(B1 / np.sqrt(mu0_n * rho0)) * np.sin(k * z - omega * t)

Ek = 0.5 * rho0 * dv**2  # Kinetic energy density
EB = dB**2 / (2 * mu0_n)  # Magnetic energy density

axes[0].plot(z, Ek, 'r-', lw=2.5, label='$E_k = \\frac{1}{2}\\rho_0 (\\delta v)^2$')
axes[0].plot(z, EB, 'b--', lw=2.5, label='$E_B = \\frac{(\\delta B)^2}{2\\mu_0}$')
axes[0].plot(z, Ek + EB, 'k-', lw=1.5, alpha=0.5, label='$E_{total}$')
axes[0].set_ylabel('Energy density', fontsize=13)
axes[0].set_xlabel('z', fontsize=13)
axes[0].set_title('Energy Densities at t=0 / t=0에서의 에너지 밀도', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].text(0.7, max(Ek) * 0.8,
             f'max(Ek) = {max(Ek):.6f}\nmax(EB) = {max(EB):.6f}\nRatio = {max(Ek)/max(EB):.6f}',
             fontsize=11, bbox=dict(boxstyle='round', fc='lightyellow'))

# --- Bottom: Integrated energies over time ---
total_Ek = []
total_EB = []
for t in t_values:
    dB_t = B1 * np.sin(k * z - omega * t)
    dv_t = -(B1 / np.sqrt(mu0_n * rho0)) * np.sin(k * z - omega * t)
    total_Ek.append(np.trapz(0.5 * rho0 * dv_t**2, z))
    total_EB.append(np.trapz(dB_t**2 / (2 * mu0_n), z))

axes[1].plot(t_values, total_Ek, 'r-', lw=2.5, label='$\\int E_k \\, dz$')
axes[1].plot(t_values, total_EB, 'b--', lw=2.5, label='$\\int E_B \\, dz$')
axes[1].plot(t_values, np.array(total_Ek) + np.array(total_EB),
             'k-', lw=1.5, alpha=0.5, label='Total')
axes[1].set_xlabel('Time', fontsize=13)
axes[1].set_ylabel('Integrated energy', fontsize=13)
axes[1].set_title('Total Energy vs Time / 총 에너지 vs 시간', fontsize=13)
axes[1].legend(fontsize=11)

plt.suptitle("Energy Equipartition in Alfvén Waves\n"
             "Alfvén 파의 에너지 등분배", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("Verification / 검증:")
print(f"  ∫Ek dz = {total_Ek[0]:.8f}")
print(f"  ∫EB dz = {total_EB[0]:.8f}")
print(f"  Ratio Ek/EB = {total_Ek[0]/total_EB[0]:.8f}")
print(f"  → Perfect equipartition (ratio = 1.0) ✓")
print(f"  Total energy is constant in time ✓ (no dissipation in ideal MHD)")

## Part 6: Alfvén Speed Across Solar Environments
## 파트 6: 태양 환경별 Alfvén 속도 비교

태양의 다양한 영역에서 Alfvén 속도를 계산합니다.
코로나에서 $v_A$가 극단적으로 빠른 것이 코로나 가열 문제에서 Alfvén 파가 주목받는 이유입니다.

Compute Alfvén speed across different solar regions.
The extremely fast $v_A$ in the corona is why Alfvén waves are prominent in the coronal heating problem.

In [ ]:
# Solar environments: name, B (T), rho (kg/m³), description
solar_envs = [
    ("Deep interior\n(Alfvén's value)\n태양 깊은 내부", 1.5e-3, 5.0,
     "Alfvén's paper"),
    ("Convection zone\n대류층", 0.01, 100,
     "Below surface"),
    ("Photosphere\n광구", 0.01, 2e-4,
     "Quiet Sun"),
    ("Sunspot umbra\n흑점 암부", 0.3, 2e-4,
     "Strong field"),
    ("Chromosphere\n채층", 5e-3, 1e-7,
     "Transition"),
    ("Quiet corona\n조용한 코로나", 1e-3, 1e-12,
     "1 MK plasma"),
    ("Active corona\n활동 코로나", 0.01, 1e-11,
     "Above AR"),
    ("Solar wind\n(1 AU)\n태양풍", 5e-9, 8e-21,
     "Near Earth"),
]

names, B_vals, rho_vals, descs = [], [], [], []
va_vals = []
for name, B, rho, desc in solar_envs:
    va = alfven_speed_si(B, rho) / 1e3  # Convert to km/s
    va_vals.append(va)
    names.append(name)
    B_vals.append(B)
    rho_vals.append(rho)
    descs.append(desc)

fig, axes = plt.subplots(1, 3, figsize=(18, 8))

# --- Left: Alfvén speed bar chart ---
colors_bar = plt.cm.coolwarm(np.linspace(0.1, 0.9, len(names)))
bars = axes[0].barh(range(len(names)), va_vals, color=colors_bar, edgecolor='black')
axes[0].set_yticks(range(len(names)))
axes[0].set_yticklabels(names, fontsize=10)
axes[0].set_xlabel('Alfvén Speed (km/s)', fontsize=12)
axes[0].set_xscale('log')
axes[0].set_title('Alfvén Speed by Region\n영역별 Alfvén 속도', fontsize=13)
# Add value labels
for i, (v, bar) in enumerate(zip(va_vals, bars)):
    axes[0].text(v * 1.5, i, f'{v:.1f} km/s', va='center', fontsize=9)

# --- Middle: B vs ρ scatter with vA contours ---
ax = axes[1]
B_range = np.logspace(-10, 0, 200)
rho_range = np.logspace(-22, 3, 200)
B_grid, rho_grid = np.meshgrid(B_range, rho_range)
va_grid = B_grid / np.sqrt(MU_0 * rho_grid) / 1e3  # km/s

contour_levels = [0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000]
cs = ax.contour(B_grid, rho_grid, va_grid, levels=contour_levels,
                colors='gray', linewidths=0.8)
ax.clabel(cs, fmt='%.3g km/s', fontsize=8)

# Plot solar environments
for i, (name, B, rho) in enumerate(zip(names, B_vals, rho_vals)):
    short_name = name.split('\n')[0]
    ax.scatter(B, rho, s=100, zorder=5, edgecolor='black', lw=1.5,
               color=colors_bar[i])
    ax.annotate(short_name, (B, rho), fontsize=8, ha='left',
                xytext=(5, 5), textcoords='offset points')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('B (T)', fontsize=12)
ax.set_ylabel('ρ (kg/m³)', fontsize=12)
ax.set_title('B-ρ Parameter Space\nB-ρ 매개변수 공간', fontsize=13)

# --- Right: Energy flux comparison ---
# Alfvén wave energy flux: F = ρ * vA * (δv)² ≈ (δB)² * vA / μ0
# Assume δB/B0 = 0.1 (10% perturbation)
ax = axes[2]
delta_frac = 0.1
fluxes = []
for B, rho in zip(B_vals, rho_vals):
    va = B / np.sqrt(MU_0 * rho)
    dB = delta_frac * B
    flux = dB**2 * va / MU_0  # W/m²
    fluxes.append(flux)

bars2 = ax.barh(range(len(names)), fluxes, color=colors_bar, edgecolor='black')
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=10)
ax.set_xlabel('Energy Flux (W/m²)', fontsize=12)
ax.set_xscale('log')
ax.set_title('Alfvén Wave Energy Flux\n(δB/B₀ = 10%)\n에너지 플럭스', fontsize=13)

# Mark coronal heating requirement
ax.axvline(300, color='red', ls='--', lw=2, label='Coronal heating\nrequirement\n~300 W/m²')
ax.legend(fontsize=9, loc='lower right')

plt.suptitle("Alfvén Waves Across the Sun\n태양 전역의 Alfvén 파", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("Key insight / 핵심 통찰:")
print("• Corona: vA ~ 1000-10000 km/s — waves carry huge energy!")
print("• Coronal heating requires ~300 W/m² — Alfvén waves can supply this")
print("• This is why Alfvén waves are a leading coronal heating candidate")

## Part 7: Frozen-in Condition Visualization
## 파트 7: Frozen-in 조건 시각화

완전 전도체에서 $\mathbf{E} + \mathbf{v} \times \mathbf{B} = 0$이므로, 자기장 선은 유체에 "얼어붙어" 함께 움직입니다.
유체가 이동하면 자기장 선도 따라 이동하고, 유체가 회전하면 자기장 선도 감깁니다.

In a perfect conductor, $\mathbf{E} + \mathbf{v} \times \mathbf{B} = 0$, so field lines are "frozen into" the fluid.
If the fluid moves, field lines follow; if the fluid rotates, field lines wind up.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# --- Row 1: Frozen-in with translational flow ---
# Three snapshots: initial, sheared, further sheared
for col, (t_label, shear_amount) in enumerate([
    ('t = 0\n(Initial / 초기)', 0),
    ('t = 1\n(Sheared / 전단)', 0.3),
    ('t = 2\n(More shear / 강한 전단)', 0.6)
]):
    ax = axes[0, col]
    y_lines = np.linspace(-1, 1, 15)
    x_range = np.linspace(-1, 1, 50)

    for y0 in y_lines:
        # Shear flow: vx = shear_amount * y
        x_displaced = x_range + shear_amount * y0
        y_vals = np.full_like(x_range, y0)
        ax.plot(x_displaced, y_vals, 'b-', lw=1.0, alpha=0.6)

    # Velocity arrows (shear flow)
    if shear_amount > 0:
        for y_arr in [-0.7, -0.3, 0.3, 0.7]:
            vx = shear_amount * y_arr * 1.5
            ax.annotate('', xy=(vx + 0.1, y_arr),
                        xytext=(-0.1, y_arr),
                        arrowprops=dict(arrowstyle='->', color='red', lw=2))

    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.2, 1.2)
    ax.set_aspect('equal')
    ax.set_title(t_label, fontsize=12)
    if col == 0:
        ax.set_ylabel('Shear Flow\n전단 흐름', fontsize=12, fontweight='bold')

# --- Row 2: Frozen-in with localized displacement (Alfvén wave) ---
for col, (t_label, phase) in enumerate([
    ('t = 0', 0),
    ('t = T/4', np.pi / 2),
    ('t = T/2', np.pi)
]):
    ax = axes[1, col]
    z_range = np.linspace(0, 2, 100)
    n_lines = 15

    for i in range(n_lines):
        x_base = -0.6 + i * 1.2 / n_lines
        # Alfvén wave displacement
        displacement = 0.15 * np.sin(2 * np.pi * z_range - phase)
        ax.plot(x_base + displacement, z_range, 'b-', lw=1.0, alpha=0.6)

    # Highlight one line
    displacement_hl = 0.15 * np.sin(2 * np.pi * z_range - phase)
    ax.plot(displacement_hl, z_range, 'r-', lw=2.5)

    # Fluid element markers on highlighted line
    for z_mark in [0.25, 0.75, 1.25, 1.75]:
        dx = 0.15 * np.sin(2 * np.pi * z_mark - phase)
        ax.plot(dx, z_mark, 'ro', ms=8, zorder=5)
        # Velocity arrow (perpendicular to B0)
        vx = -0.15 * 2 * np.pi * np.cos(2 * np.pi * z_mark - phase)
        ax.annotate('', xy=(dx + vx * 0.05, z_mark),
                    xytext=(dx, z_mark),
                    arrowprops=dict(arrowstyle='->', color='green', lw=2))

    ax.set_xlim(-0.8, 0.8)
    ax.set_ylim(-0.1, 2.1)
    ax.set_title(t_label, fontsize=12)
    ax.set_xlabel('x (transverse)', fontsize=10)
    if col == 0:
        ax.set_ylabel('Alfvén Wave\nAlfvén 파\n\nz (along B₀)', fontsize=12,
                       fontweight='bold')

# Add legend explanation
axes[1, 2].text(0.4, 1.0, '● Fluid\n   elements\n   유체 요소\n\n→ Velocity\n   속도',
                fontsize=10, transform=axes[1, 2].transAxes,
                bbox=dict(boxstyle='round', fc='lightyellow'))

plt.suptitle("Frozen-in Condition: Field Lines Move with the Fluid\n"
             "Frozen-in 조건: 자기장 선이 유체와 함께 이동",
             fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("Top row: Shear flow drags field lines → builds magnetic tension")
print("Bottom row: Alfvén wave — fluid elements oscillate transversely,")
print("  field lines follow (frozen-in), tension provides restoring force")

## Part 8: Three MHD Wave Modes — Modern Context
## 파트 8: 세 가지 MHD 파동 모드 — 현대적 맥락

Alfvén이 발견한 파동은 실제로 MHD에서 존재하는 **세 가지 파동 모드** 중 하나입니다:

The wave Alfvén discovered is one of **three wave modes** in MHD:

1. **Alfvén wave** (shear/torsional): 비압축성 횡파, $v = v_A \cos\theta$ / Incompressible transverse, $v = v_A \cos\theta$
2. **Fast magnetosonic wave**: 자기압 + 가스압이 복원력, $v > v_A$ / Magnetic + gas pressure restoring, $v > v_A$
3. **Slow magnetosonic wave**: 자기장 vs 가스의 경쟁, $v < \min(v_A, c_s)$ / Field vs gas competition, $v < \min(v_A, c_s)$

Friedrichs diagram(파동 속도의 극좌표 분포)으로 세 모드를 비교합니다.

We compare all three modes using a Friedrichs diagram (polar plot of phase speed).

In [ ]:
def mhd_phase_speeds(theta: np.ndarray, va: float, cs: float) -> tuple:
    """Compute phase speeds for the three MHD wave modes.

    Args:
        theta: Propagation angle relative to B0 (radians).
        va: Alfvén speed.
        cs: Sound speed.

    Returns:
        Tuple of (v_fast, v_alfven, v_slow) arrays.
    """
    # Alfvén wave
    v_alfven = va * np.abs(np.cos(theta))

    # Fast and slow magnetosonic
    va2 = va**2
    cs2 = cs**2
    cos2 = np.cos(theta)**2

    discriminant = np.sqrt((va2 + cs2)**2 - 4 * va2 * cs2 * cos2)
    v_fast = np.sqrt(0.5 * (va2 + cs2 + discriminant))
    v_slow = np.sqrt(np.maximum(0, 0.5 * (va2 + cs2 - discriminant)))

    return v_fast, v_alfven, v_slow


theta = np.linspace(0, 2 * np.pi, 500)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), subplot_kw={'projection': 'polar'})

# Three plasma beta regimes
configs = [
    (1.0, 0.5, 'Low β (corona)\n저 β (코로나)\n$v_A > c_s$'),
    (1.0, 1.0, 'β = 1\n$v_A = c_s$'),
    (0.5, 1.0, 'High β (interior)\n고 β (내부)\n$v_A < c_s$'),
]

for ax, (va, cs, title) in zip(axes, configs):
    v_fast, v_alfven, v_slow = mhd_phase_speeds(theta, va, cs)

    ax.plot(theta, v_fast, 'r-', lw=2.5, label=f'Fast (빠른 모드)')
    ax.plot(theta, v_alfven, 'b-', lw=2.5, label=f'Alfvén')
    ax.plot(theta, v_slow, 'g-', lw=2.5, label=f'Slow (느린 모드)')

    # Mark B0 direction
    ax.annotate('B₀', xy=(0, ax.get_ylim()[1] * 0.85), fontsize=14,
                fontweight='bold', color='black', ha='center')

    ax.set_title(title, fontsize=12, pad=20)
    ax.legend(fontsize=9, loc='lower right', bbox_to_anchor=(1.3, -0.1))

    # Set limits
    max_v = max(va, cs) * 1.6
    ax.set_ylim(0, max_v)

plt.suptitle("Friedrichs Diagram: Three MHD Wave Modes\n"
             "Friedrichs 다이어그램: 세 가지 MHD 파동 모드\n"
             "(Phase speed as function of propagation angle θ / 전파 각도 θ에 따른 위상 속도)",
             fontsize=14, y=1.08)
plt.tight_layout()
plt.show()

print("Alfvén's 1942 paper predicted the Alfvén (shear) mode:")
print("• Propagates ONLY along B₀ (cos θ dependence)")
print("• v = vA at θ=0, v = 0 at θ=90° (no perpendicular propagation)")
print("• Fast mode: propagates in all directions, v ≥ max(vA, cs)")
print("• Slow mode: propagates near B₀, v ≤ min(vA, cs)")
print("\nIn the corona (low β): Alfvén and fast modes dominate")
print("In the interior (high β): slow mode becomes important")

## Summary / 요약

| Part | 내용 / Content | Alfvén (1942) 연결 / Connection |
|---|---|---|
| 1 | Alfvén 속도 계산 (CGS + SI) | 논문의 수치($V \sim 60$ cm/s) 정확히 재현 |
| 2 | 해석해 시각화 (정현파) | 파동 방정식의 횡파 해 — 자기장 선이 "흔들리는" 모습 |
| 3 | 1D 수치 시뮬레이션 (Leapfrog) | 가우시안 펄스가 $v_A$로 전파 — 비분산 확인 |
| 4 | 기타 줄 비유 | 자기장력 ↔ 줄 장력의 물리적 대응 |
| 5 | 에너지 등분배 검증 | $E_k = E_B$ — Alfvén 파의 본질적 특성 |
| 6 | 태양 환경별 $v_A$ | 코로나에서 $v_A \sim 10^4$ km/s → 코로나 가열 후보 |
| 7 | Frozen-in 조건 | 완전 전도체에서 자기장 선이 유체에 얼어붙는 물리 |
| 8 | 세 가지 MHD 모드 (Friedrichs) | Alfvén 파는 MHD 파동의 하나 — 현대적 맥락 |

**다음 논문 / Next paper**: Babcock (1961) — 태양 다이나모 모델.
Alfvén의 MHD가 Hale의 극성 법칙(SP #7)과 결합되면, 태양 자기 주기를 설명하는 **다이나모 이론**이 탄생합니다.

**Next paper**: Babcock (1961) — Solar dynamo model.
When Alfvén's MHD combines with Hale's polarity law (SP #7), the **dynamo theory** explaining the solar magnetic cycle is born.